# My C++ mixed-volume experiment
I implement the same cube-and-ball framework in C++17. I use a Python notebook as an interface for compiling and recording its output. The numerical computation below runs in C++, not in Python. I do not claim certified mixing or ideal-oracle accuracy.

I edit the settings in the next cell. Dimension means coordinates; lambda is the added ball radius; k is the coefficient index; M is the standalone B3 label count; samples is the per-phase/query budget; burn-in discards initial moves; spacing is the number of moves before a recorded point; threshold is a desired component probability, not error tolerance; seed controls randomness; tolerance controls chord bisection. These are editable example values.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "cpp/mixed_volume_experiment.cpp").exists():
    raise RuntimeError("Open this notebook from the repository root or notebooks directory.")

import subprocess, shutil
compiler = shutil.which("clang++") or shutil.which("g++") or shutil.which("c++")
if compiler is None:
    raise RuntimeError("Install a C++17 compiler first.")
binary = ROOT / "results/mixed_volume_benchmark"
subprocess.run([compiler,"-std=c++17","-O2",str(ROOT/"cpp/mixed_volume_experiment.cpp"),"-o",str(binary)],check=True)
print(subprocess.check_output([compiler,"--version"],text=True))


g++ (Ubuntu 13.3.0-6ubuntu2~24.04) 13.3.0
Copyright (C) 2023 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.




In [2]:
N = 3
LAMBDA = 1.0
K = 2
M = 2000
SAMPLES = 5000
BURN_IN = 2000
SPACING = 3
THRESHOLD = 0.05
SEED = 2026
TOLERANCE = 1e-9
settings = [N,LAMBDA,K,M,SAMPLES,BURN_IN,SPACING,THRESHOLD,SEED,TOLERANCE]
completed = subprocess.run([str(binary)],input="\n".join(map(str,settings))+"\n",capture_output=True,text=True,check=True)
print(completed.stdout)


Experimental mixed-volume oracles for a centered cube plus a Euclidean ball.

Dimension is the number of coordinates per point.
Choose n (positive integer): 
Lambda is the radius of the ball added to the cube; zero leaves the cube unchanged.
Choose lambda: 
The target index labels the mixed-volume coefficient; 0 means cube volume and n means ball volume.
Choose k (0 through n): 
This is the number of labels recorded for the standalone B3 distribution.
Choose m: 
This separate budget is used per B1 phase and initial Algorithm 2 query.
Choose samples: 
Burn-in is the number of initial moves discarded; it is not a certified mixing time.
Choose burn-in: 
Spacing is the number of moves before each recorded point; it does not guarantee independence.
Choose spacing: 
The threshold is a desired component probability for the search, not an error tolerance.
Choose threshold (strictly between 0 and 1): 
A seed controls the random sequence; C++ samples need not match Python samples.
Choose nonnega

## How I read my output
I compare standalone B1 at the input lambda with its exact cube volume. Algorithm 3 runs Algorithm 2 and may select a different lambda. I compare the final mixed-volume estimate against its exact cube coefficient. My observed error for one seed is not an accuracy guarantee. I use the comparison notebook for matched timing rather than timing this compilation and notebook interface.